# AgentCore Optimization 튜토리얼 — HR Assistant

이 노트북에서는 HR Assistant 에이전트를 사용하여 전체 **AgentCore Optimization** 워크플로를 살펴봅니다.

1. HR Assistant를 AgentCore Runtime에 **배포**합니다.
2. 배치 평가로 기준 성능을 **평가**합니다.
3. 시스템 프롬프트와 도구 설명의 개선안을 **추천**받습니다.
4. Configuration Bundle을 사용하여 모델 ID, 시스템 프롬프트, 도구 설명 설정을 **묶습니다**.
5. **A/B 테스트** — Configuration Bundle 라우팅으로 기존 번들과 새 번들을 비교합니다.
6. **A/B 테스트** — Target-Based 라우팅으로 두 에이전트 엔드포인트를 비교합니다.
7. 생성한 모든 리소스를 **정리**합니다.

위에서 아래 순서로 **Run All Cells**를 실행하세요. 각 섹션은 이전 섹션의 결과를 사용합니다.

---

## 사전 요구 사항

- Bedrock AgentCore 액세스가 활성화된 AWS 계정
- 설정된 AWS 자격 증명(`aws configure` 또는 환경 변수)
- IAM 권한: `bedrock-agentcore:*`, `bedrock:InvokeModel`, `iam:*`, `s3:*`, `logs:*`, `xray:*`, `ecr:*`
- 로컬에서 실행 중인 Docker(에이전트 컨테이너 빌드용)
- Step 0의 패키지가 설치된 Python 3.10 이상

> **참고:** 에이전트를 호출한 후 CloudWatch에 데이터가 수집되기까지 2~3분이 걸립니다. 배치 평가는 1~5분, 권장 사항 생성은 2~5분이 걸립니다. 전체 노트북을 완료하는 데 약 45분이 필요합니다.

## Step 0: 종속성 설치

In [ ]:
# 필수 패키지 설치
# bedrock-agentcore: Runtime 호출, 평가, 권장 사항, A/B 테스트용 Python SDK
# boto3: AWS SDK(v1.43 이상에는 공개 AgentCore API 포함)
# requests + botocore: SigV4로 서명된 Gateway 호출용
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "bedrock-agentcore>=1.7.0",
        "boto3>=1.43.0",
        "requests",
        "--quiet",
    ],
    check=True,
)
print("Dependencies installed.")

## Step 1: 설정

리전과 고유한 접미사를 설정합니다. 접미사는 노트북을 다시 실행할 때 이름 충돌을 방지합니다.

In [ ]:
import boto3
import json
import uuid
import time
import subprocess
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── 설정 ─────────────────────────────────────────────────────────
REGION = "us-east-1"  # 원하는 리전으로 변경
SUFFIX = uuid.uuid4().hex[:6]  # 이름 충돌을 방지하는 고유 접미사

# 노트북을 여러 번 실행할 수 있도록 Runtime 이름에 접미사 사용
V1_NAME = f"HRAssistV1{SUFFIX}"
V2_NAME = f"HRAssistV2{SUFFIX}"

# 계정 자동 감지
sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# ── boto3 client 설정 ─────────────────────────────────────────────────────
# 제어 영역: Gateway, Runtime, Configuration Bundle, Online Evaluation Config
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)
# 데이터 영역: 에이전트 호출, 배치 평가, 권장 사항, A/B 테스트
dp = boto3.client("bedrock-agentcore", region_name=REGION)
# 지원 서비스
iam = boto3.client("iam", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)

print(f"ACCOUNT_ID = {ACCOUNT_ID}")
print(f"REGION     = {REGION}")
print(f"SUFFIX     = {SUFFIX}")
print(f"V1_NAME    = {V1_NAME}")
print(f"V2_NAME    = {V2_NAME}")

## Step 2: HR Assistant(v1) 배포

`deploy_agent.py` 스크립트는 다음 작업을 수행합니다.
1. IAM 실행 역할 생성
2. `hr_assistant_agent.py`와 ARM64 종속성 패키징
3. S3에 업로드
4. AgentCore Runtime을 생성하고 `ACTIVE` 상태가 될 때까지 폴링
5. 상태를 `agent_state_{name}.json`에 저장

**IAM 역할 설정:**
권한 정책은 `bedrock-agentcore:*`(A/B 테스트 서비스에 필요한 Gateway, Configuration Bundle, Online Evaluation 읽기 포함)와 점수 집계에 필요한 CloudWatch Logs 작업인 `DescribeLogGroups`, `DescribeIndexPolicies`, `PutIndexPolicy`, `StartQuery`, `GetQueryResults`, `StopQuery`, `FilterLogEvents`, `GetLogEvents`를 허용합니다.

서비스가 컨테이너 이미지를 빌드하는 동안 일반적으로 **3~5분**이 걸립니다.

In [ ]:
# HR Assistant v1(기준 에이전트) 배포
result = subprocess.run(
    [
        sys.executable,
        "deploy_agent.py",
        "--name",
        V1_NAME,
        "--region",
        REGION,
        "--version",
        "v1",
    ],
    capture_output=False,  # 출력을 노트북으로 스트리밍
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"deploy_agent.py failed with exit code {result.returncode}")

In [ ]:
# 배포된 상태 불러오기
v1_state = json.loads(Path(f"agent_state_{V1_NAME}.json").read_text())

AGENT_ARN = v1_state["runtime_arn"]
AGENT_ID = v1_state["runtime_id"]
LOG_GROUP = v1_state["log_group"]
SERVICE_NAME = v1_state["service_name"]
ROLE_ARN = v1_state["role_arn"]
ROLE_NAME = v1_state["role_name"]
S3_BUCKET = v1_state["s3_bucket"]

# OTel span도 공유 'aws/spans' 로그 그룹에 저장됨
SPANS_LOG_GROUP = "aws/spans"

LOG_GROUP_ARN = f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP}"
SPANS_LOG_ARN = f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{SPANS_LOG_GROUP}"

print(f"Agent ARN    : {AGENT_ARN}")
print(f"Log Group    : {LOG_GROUP}")
print(f"Service Name : {SERVICE_NAME}")

## Step 3: 기준 Configuration Bundle 생성 및 트래픽 전송

**Configuration Bundle**은 Runtime ARN을 키로 사용하는 버전 관리형 에이전트 구성 컨테이너입니다. 에이전트는 호출 시 번들에서 `system_prompt`와 `tool_descriptions` 같은 다른 키를 읽으므로 다시 배포할 필요가 없습니다.

각 번들 호출은 `bundleId`(고정 식별자)와 `versionId`(변경할 수 없는 스냅샷)를 반환합니다. Git 커밋처럼 `commitMessage`를 사용하여 변경 내용을 기록하세요.

현재 시스템 프롬프트와 도구 설명으로 **기준 번들**을 만든 다음, 대표적인 HR 세션 10개를 전송합니다. 이 추적 데이터는 이후 단계의 기준 평가와 권장 사항 생성에 사용됩니다.

In [ ]:
CURRENT_SYSTEM_PROMPT = """You are a helpful HR Assistant for Acme Corp.

You help employees with:
- Checking PTO (paid time off) balances
- Submitting PTO requests
- Looking up HR policies (PTO, remote work, parental leave, code of conduct)
- Understanding employee benefits (health, dental, vision, 401k, life insurance)
- Retrieving pay stub information

Always use the available tools to answer questions accurately. Do not make up
policy details, benefit amounts, or pay information — look them up.
Be concise, professional, and friendly."""

CURRENT_TOOL_DESCRIPTIONS = {
    "get_pto_balance": "Return the current PTO balance for an employee.",
    "submit_pto_request": "Submit a PTO request for an employee.",
    "lookup_hr_policy": "Look up a company HR policy document by topic.",
    "get_benefits_summary": "Return a summary of a specific employee benefit.",
    "get_pay_stub": "Retrieve a pay stub for an employee for a specific pay period.",
}

# 기준 Configuration Bundle 생성
# commitMessage는 Git commit처럼 이 버전에 포함된 내용을 기록함
baseline_resp = ctrl.create_configuration_bundle(
    bundleName=f"HRBaseline{SUFFIX}",
    description="HR Assistant baseline configuration",
    components={
        AGENT_ARN: {
            "configuration": {
                "system_prompt": CURRENT_SYSTEM_PROMPT,
                "tool_descriptions": CURRENT_TOOL_DESCRIPTIONS,
            }
        }
    },
    commitMessage="Initial configuration — baseline system prompt and tool descriptions",
    clientToken=str(uuid.uuid4()),
)
BASELINE_BUNDLE_ARN = baseline_resp["bundleArn"]
BASELINE_BUNDLE_VERSION = baseline_resp["versionId"]
BASELINE_BUNDLE_ID = baseline_resp["bundleId"]

baseline_baggage = (
    f"aws.agentcore.configbundle_arn={BASELINE_BUNDLE_ARN},aws.agentcore.configbundle_version={BASELINE_BUNDLE_VERSION}"
)

print(f"Baseline bundle ID : {BASELINE_BUNDLE_ID}")
print(f"Version            : {BASELINE_BUNDLE_VERSION}")

In [ ]:
# 기준 트래픽을 위한 대표 HR 시나리오
BASELINE_PROMPTS = [
    ("EMP-001", "What is my current PTO balance?"),
    (
        "EMP-001",
        "Please submit a PTO request for me from 2026-06-01 to 2026-06-05 for a family vacation.",
    ),
    ("EMP-001", "Can you pull up my January 2026 pay stub?"),
    ("EMP-002", "How many PTO days do I have left? I only joined recently."),
    ("EMP-042", "What's the company policy on working from home?"),
    (
        "EMP-001",
        "What are my health insurance options and how much does the company cover?",
    ),
    ("EMP-042", "Tell me about the 401k plan — how much does the company match?"),
    ("EMP-001", "What is the parental leave policy for primary caregivers?"),
    (
        "EMP-002",
        "I want to request time off from 2026-07-14 to 2026-07-18 for a medical procedure.",
    ),
    (
        "EMP-042",
        "Can you show me my December 2025 pay stub and explain the deductions?",
    ),
]

baseline_session_ids = []
for emp_id, prompt in BASELINE_PROMPTS:
    session_id = str(uuid.uuid4())
    baseline_session_ids.append(session_id)
    full_prompt = f"Employee ID: {emp_id}. {prompt}"
    resp = dp.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": full_prompt}).encode(),
        baggage=baseline_baggage,
    )
    response_text = resp["response"].read().decode("utf-8")
    print(f"Session {session_id[:8]}... [{emp_id}] {prompt[:55]}")

print(f"\nSent {len(baseline_session_ids)} baseline sessions.")
print("Waiting 3 minutes for CloudWatch ingestion...")
for remaining in range(180, 0, -30):
    print(f"  {remaining}s remaining...")
    time.sleep(30)
print("CloudWatch ingestion complete.")

## Step 4: 기준 배치 평가

**배치 평가**는 현재 에이전트의 성능을 측정합니다. CloudWatch에서 세션을 찾아 각각 내장 LLM 평가기로 평가하고 집계 점수를 반환합니다.

| 평가기 | 측정 항목 |
|-----------|------------------|
| **GoalSuccessRate** | 에이전트가 사용자의 목표를 달성했는가? |
| **Helpfulness** | 응답이 유용하고 실행 가능한가? |
| **Correctness** | Agent가 정확한 정보를 제공했는가? |

이 점수는 A/B 테스트에서 넘어야 할 **기준값**이 됩니다.

In [ ]:
# 기준 배치 평가 시작
# Public API: batchEvaluationName, 최상위 evaluators, dataSourceConfig
eval_resp = dp.start_batch_evaluation(
    batchEvaluationName=f"HRBaseline{SUFFIX}",
    evaluators=[
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.Helpfulness"},
        {"evaluatorId": "Builtin.Correctness"},
    ],
    dataSourceConfig={
        "cloudWatchLogs": {
            "serviceNames": [SERVICE_NAME],
            "logGroupNames": [SPANS_LOG_GROUP, LOG_GROUP],
            "filterConfig": {"sessionIds": baseline_session_ids},
        }
    },
    clientToken=str(uuid.uuid4()),
)
BASELINE_EVAL_ID = eval_resp["batchEvaluationId"]
print(f"Started batch evaluation: {BASELINE_EVAL_ID}")
print("Polling for completion (typically 2–5 minutes)...")

In [ ]:
# 종료 상태가 될 때까지 폴링
TERMINAL = {"COMPLETED", "FAILED", "STOPPED", "COMPLETED_WITH_ERRORS"}
while True:
    result = dp.get_batch_evaluation(batchEvaluationId=BASELINE_EVAL_ID)
    status = result["status"]
    print(f"  Status: {status}")
    if status in TERMINAL:
        break
    time.sleep(30)

# API 응답을 먼저 사용하고, 없으면 CloudWatch로 대체
baseline_scores = {}
er = result.get("evaluationResults", {})
for s in er.get("evaluatorSummaries", []):
    avg = s.get("statistics", {}).get("averageScore")
    if avg is not None:
        baseline_scores[s["evaluatorId"]] = avg

if not baseline_scores:
    print("  Reading scores from CloudWatch...")
    baseline_scores = fetch_eval_scores(BASELINE_EVAL_ID)

print(f"\n{'Evaluator':<35} {'Score':>8}")
print("-" * 45)
for eid, score in sorted(baseline_scores.items()):
    print(f"{eid:<35} {score:>8.4f}")
print("\nBaseline scores saved.")

## Step 5: 최적화 권장 사항

AgentCore는 프로덕션 추적 데이터를 분석하여 개선된 에이전트 구성을 생성합니다. 다음 두 가지 유형을 사용할 수 있습니다.

- **시스템 프롬프트 권장 사항**: 목표 지표를 개선하도록 시스템 프롬프트를 다시 작성합니다.
- **도구 설명 권장 사항**: 에이전트가 올바른 도구를 더 자주 선택하도록 도구 설명을 개선합니다.

### 5a: 시스템 프롬프트 권장 사항

In [ ]:
now = datetime.now(timezone.utc)
start_dt = now - timedelta(days=7)

sp_rec_resp = dp.start_recommendation(
    name=f"HRSpRec{SUFFIX}",
    type="SYSTEM_PROMPT_RECOMMENDATION",
    recommendationConfig={
        "systemPromptRecommendationConfig": {
            "systemPrompt": {"text": CURRENT_SYSTEM_PROMPT},
            "agentTraces": {
                "cloudwatchLogs": {
                    "logGroupArns": [LOG_GROUP_ARN],
                    "serviceNames": [SERVICE_NAME],
                    "startTime": start_dt,
                    "endTime": now,
                }
            },
            "evaluationConfig": {
                "evaluators": [{"evaluatorArn": "arn:aws:bedrock-agentcore:::evaluator/Builtin.GoalSuccessRate"}]
            },
        }
    },
    clientToken=str(uuid.uuid4()),
)
SP_REC_ID = sp_rec_resp["recommendationId"]
print(f"Started system prompt recommendation: {SP_REC_ID}")
print("Polling (typically 2–5 minutes)...")

In [ ]:
REC_TERMINAL = {"COMPLETED", "FAILED"}
while True:
    sp_result = dp.get_recommendation(recommendationId=SP_REC_ID)
    status = sp_result["status"]
    print(f"  Status: {status}")
    if status in REC_TERMINAL:
        break
    time.sleep(30)

rec = sp_result.get("recommendationResult", {})
sp_rec_result = rec.get("systemPromptRecommendationResult", {})
RECOMMENDED_SYSTEM_PROMPT = sp_rec_result.get("recommendedSystemPrompt") or CURRENT_SYSTEM_PROMPT

if sp_rec_result.get("errorCode"):
    print(f"Recommendation error ({sp_rec_result['errorCode']}): {sp_rec_result.get('errorMessage', '')[:200]}")
    print("Falling back to current system prompt.")

print("\n" + "=" * 60)
print("RECOMMENDED SYSTEM PROMPT")
print("=" * 60)
print(RECOMMENDED_SYSTEM_PROMPT)

### 5b: 도구 설명 권장 사항

도구 설명 권장 사항은 LLM이 올바른 도구를 더 안정적으로 선택하도록 에이전트의 도구 설명을 개선합니다. 개선안은 검토할 수 있도록 표시되며, 이후 에이전트 코드 업데이트에 적용하거나 Configuration Bundle에 저장할 수 있습니다.

In [ ]:
tools_list = [{"toolName": name, "toolDescription": {"text": desc}} for name, desc in CURRENT_TOOL_DESCRIPTIONS.items()]

td_rec_resp = dp.start_recommendation(
    name=f"HRTdRec{SUFFIX}",
    type="TOOL_DESCRIPTION_RECOMMENDATION",
    recommendationConfig={
        "toolDescriptionRecommendationConfig": {
            "toolDescription": {"toolDescriptionText": {"tools": tools_list}},
            "agentTraces": {
                "cloudwatchLogs": {
                    "logGroupArns": [LOG_GROUP_ARN],
                    "serviceNames": [SERVICE_NAME],
                    "startTime": start_dt,
                    "endTime": now,
                }
            },
        }
    },
    clientToken=str(uuid.uuid4()),
)
TD_REC_ID = td_rec_resp["recommendationId"]
print(f"Started tool description recommendation: {TD_REC_ID}")
print("Polling (typically 2–5 minutes)...")

In [ ]:
while True:
    td_result = dp.get_recommendation(recommendationId=TD_REC_ID)
    status = td_result["status"]
    print(f"  Status: {status}")
    if status in REC_TERMINAL:
        break
    time.sleep(30)

RECOMMENDED_TOOL_DESCRIPTIONS = dict(CURRENT_TOOL_DESCRIPTIONS)  # 기본값: 변경 없음

if status == "COMPLETED":
    td_rec_result = td_result.get("recommendationResult", {}).get("toolDescriptionRecommendationResult", {})
    returned_tools = td_rec_result.get("tools", [])
    tool_keys = list(CURRENT_TOOL_DESCRIPTIONS.keys())

    if td_rec_result.get("errorCode"):
        print(f"Recommendation error ({td_rec_result['errorCode']}): {td_rec_result.get('errorMessage', '')[:200]}")
        print("Using current tool descriptions.")
    elif returned_tools:
        print("\n" + "=" * 60)
        print("RECOMMENDED TOOL DESCRIPTIONS")
        print("=" * 60)
        for i, item in enumerate(returned_tools):
            new_desc = item.get("recommendedToolDescription", "")
            tool_name = item.get("toolName") or (tool_keys[i] if i < len(tool_keys) else f"tool_{i}")
            RECOMMENDED_TOOL_DESCRIPTIONS[tool_name] = new_desc
            print(f"\n[{tool_name}]")
            print(f"  Before: {CURRENT_TOOL_DESCRIPTIONS.get(tool_name, '(unknown)')}")
            print(f"  After : {new_desc}")
    else:
        print("No tool description recommendations returned. Using current descriptions.")
else:
    print(f"Tool description recommendation status: {status}. Using current descriptions.")

## Step 6: Configuration Bundle — 생성, 업데이트, 조회, 비교

**Configuration Bundle**은 Runtime ARN을 키로 사용하는 버전 관리형 에이전트 구성 컨테이너입니다. `create` 또는 `update`를 호출할 때마다 변경할 수 없는 새 버전이 생성되므로 언제든 이전 버전을 조회하거나 롤백할 수 있습니다.

### 번들 수명 주기

| 작업 | API | 사용 시점 |
|-----------|-----|-------------|
| 생성 | `create_configuration_bundle` | 최초 생성 시 `bundleId` 설정 |
| 업데이트 | `update_configuration_bundle` | 평가 후 `parentVersionIds`를 전달하여 계보 기록 |
| 조회 | `get_configuration_bundle` | 현재 구성 확인 |
| 비교 | `get_configuration_bundle_version` | 두 버전의 차이 비교 |

생성하거나 업데이트할 때마다 `commitMessage`를 사용하여 구성을 변경한 *이유*를 기록하세요.

### 에이전트 통합

에이전트는 호출 핸들러에서 `BedrockAgentCoreContext.get_config_bundle()`을 사용하여 번들을 읽습니다. 번들은 Gateway 또는 클라이언트 코드가 설정한 baggage 헤더를 통해 자동으로 주입됩니다.

```python
bundle = BedrockAgentCoreContext.get_config_bundle()
system_prompt = DEFAULT_SYSTEM_PROMPT
tool_descs = {}
if bundle:
    system_prompt = bundle.get("system_prompt", DEFAULT_SYSTEM_PROMPT)
    tool_descs = bundle.get("tool_descriptions", {})

agent.system_prompt = system_prompt
if tool_descs:
    for t in agent.tools:
        if t.tool_name in tool_descs:
            t.tool_spec["description"] = tool_descs[t.tool_name]
```

### 여기서 생성할 항목

- **Control (C)** — 기존 시스템 프롬프트 + 기존 도구 설명  
- **Treatment (T1)** — 추천 시스템 프롬프트 + 추천 도구 설명  

생성 후 두 번들을 조회하고 비교하여 업데이트가 적용되었는지 확인합니다.

In [ ]:
# ── Step 6a: Control Bundle 생성(기존 구성) ──────────────────────
control_resp = ctrl.create_configuration_bundle(
    bundleName=f"HRControl{SUFFIX}",
    description="HR Assistant control variant — original system prompt and tool descriptions",
    components={
        AGENT_ARN: {
            "configuration": {
                "system_prompt": CURRENT_SYSTEM_PROMPT,
                "tool_descriptions": CURRENT_TOOL_DESCRIPTIONS,
            }
        }
    },
    commitMessage="Control: original system prompt and tool descriptions (v1 baseline)",
    clientToken=str(uuid.uuid4()),
)
CONTROL_BUNDLE_ARN = control_resp["bundleArn"]
CONTROL_BUNDLE_VERSION = control_resp["versionId"]
CONTROL_BUNDLE_ID = control_resp["bundleId"]
print(f"Control bundle ID      : {CONTROL_BUNDLE_ID}")
print(f"Control bundle version : {CONTROL_BUNDLE_VERSION}")

# ── Step 6b: Treatment Bundle 생성(추천 시스템 프롬프트와 도구 설명을 포함한 구성) ──────────────────
treatment_resp = ctrl.create_configuration_bundle(
    bundleName=f"HRTreatment{SUFFIX}",
    description="HR Assistant treatment variant — recommended system prompt and tool descriptions",
    components={
        AGENT_ARN: {
            "configuration": {
                "system_prompt": RECOMMENDED_SYSTEM_PROMPT,
                "tool_descriptions": RECOMMENDED_TOOL_DESCRIPTIONS,
            }
        }
    },
    commitMessage="Treatment: AI-recommended system prompt + improved tool descriptions from Step 5",
    clientToken=str(uuid.uuid4()),
)
TREATMENT_BUNDLE_ARN = treatment_resp["bundleArn"]
TREATMENT_BUNDLE_VERSION = treatment_resp["versionId"]
TREATMENT_BUNDLE_ID = treatment_resp["bundleId"]
print(f"\nTreatment bundle ID      : {TREATMENT_BUNDLE_ID}")
print(f"Treatment bundle version : {TREATMENT_BUNDLE_VERSION}")

### 6c: 번들 조회

현재 구성을 조회하여 올바르게 저장되었는지 확인합니다. `get_configuration_bundle`은 항상 **최신** 버전을 반환합니다.

In [ ]:
# ── Step 6c: Treatment Bundle 조회 ────────────────────────────────────
read_resp = ctrl.get_configuration_bundle(bundleId=TREATMENT_BUNDLE_ID)
config = read_resp["components"][AGENT_ARN]["configuration"]

print(f"Bundle ID  : {read_resp['bundleId']}")
print(f"Version    : {read_resp['versionId']}")
print("\nSystem prompt (first 200 chars):")
print(f"  {config['system_prompt'][:200]}...")
print(f"\nTool descriptions ({len(config.get('tool_descriptions', {}))} tools):")
for tool_name, desc in config.get("tool_descriptions", {}).items():
    print(f"  [{tool_name}] {desc[:100]}")

### 6d: 버전 비교

`get_configuration_bundle_version`은 ID로 특정 버전을 조회합니다. Control(기존) 버전과 Treatment(추천) 버전을 비교하여 정확한 변경 사항을 확인합니다. 감사 및 롤백 결정에 유용합니다.

In [ ]:
# ── Step 6d: Control 및 Treatment 버전 비교 ────────────────────────
# 각 versionId로 두 버전 조회
v_control = ctrl.get_configuration_bundle_version(bundleId=CONTROL_BUNDLE_ID, versionId=CONTROL_BUNDLE_VERSION)
v_treatment = ctrl.get_configuration_bundle_version(bundleId=TREATMENT_BUNDLE_ID, versionId=TREATMENT_BUNDLE_VERSION)

cfg_c = v_control["components"][AGENT_ARN]["configuration"]
cfg_t = v_treatment["components"][AGENT_ARN]["configuration"]

print("=" * 60)
print("CONTROL vs TREATMENT — Configuration Diff")
print("=" * 60)

all_keys = sorted(set(cfg_c.keys()) | set(cfg_t.keys()))
for key in all_keys:
    val_c = cfg_c.get(key)
    val_t = cfg_t.get(key)
    if val_c == val_t:
        continue  # 변경 없음

    print(f"\n[{key}]")
    if isinstance(val_c, dict) and isinstance(val_t, dict):
        # tool_descriptions의 도구별 차이 표시
        for tool in sorted(set(val_c) | set(val_t)):
            if val_c.get(tool) != val_t.get(tool):
                print(f"  Tool: {tool}")
                print(f"    Before: {str(val_c.get(tool, '(missing)'))[:120]}")
                print(f"    After : {str(val_t.get(tool, '(missing)'))[:120]}")
    else:
        c_str = str(val_c or "")
        t_str = str(val_t or "")
        print(f"  Before ({len(c_str)} chars): {c_str[:200]}")
        print(f"  After  ({len(t_str)} chars): {t_str[:200]}")

print("\nDiff complete.")
print(f"\nControl   commitMessage: {v_control.get('commitMessage', 'n/a')}")
print(f"Treatment commitMessage: {v_treatment.get('commitMessage', 'n/a')}")

## Step 7: A/B 테스트 — Configuration Bundle 라우팅

테스트할 변경 사항이 다른 시스템 프롬프트, 모델 ID 또는 도구 설명처럼 구성에만 해당하면 Configuration Bundle 라우팅을 사용합니다. 두 그룹은 서로 다른 Configuration Bundle 버전을 사용하여 같은 Runtime에서 실행됩니다. Gateway는 W3C baggage 헤더를 통해 각 요청에 올바른 번들 참조를 주입하고, 에이전트는 Runtime에서 이를 읽습니다. 따라서 **Runtime 하나**와 **Online Evaluation Config 하나**만 배포하면 됩니다.

코드 변경, 프레임워크 업그레이드 또는 완전히 다른 에이전트 구현이 포함되면 두 개의 별도 Runtime으로 트래픽을 전송하는 Target-Based 라우팅(Step 8)을 사용하세요.

**아키텍처:**
```
사용자 요청
     │
     ▼
[Gateway] ──50%──▶ [Control Bundle C]   ──┐
     │                                     ├──▶ [HR Runtime v1] ──▶ CloudWatch
     └──50%──▶ [Treatment Bundle T1] ──────┘                            │
                                                   [Online Evaluation Config] ◀┘
                                                           │
                                                   [A/B 테스트 결과]
```

세션 할당은 **고정(sticky)**됩니다. 세션 ID가 그룹에 할당되면 이후 같은 세션 ID의 모든 요청이 동일한 그룹으로 라우팅됩니다. 따라서 세션 내에서는 일관된 경험을 제공하면서 새 세션은 트래픽 가중치에 따라 그룹에 분산할 수 있습니다.

### 7a: Gateway 생성

In [ ]:
# IAM 권한 부여자를 사용하는 HTTP Gateway 생성
# Gateway가 요청을 가로채고 baggage 헤더를 통해 Configuration Bundle 주입
gw_resp = ctrl.create_gateway(
    name=f"HRGateway{SUFFIX}",
    description="HR Assistant A/B test gateway",
    authorizerType="AWS_IAM",
    roleArn=ROLE_ARN,
    clientToken=str(uuid.uuid4()),
)
GATEWAY_ID = gw_resp["gatewayId"]
print(f"Gateway created: {GATEWAY_ID}. Polling for READY...")

for i in range(30):
    gw = ctrl.get_gateway(gatewayIdentifier=GATEWAY_ID)
    gw_status = gw.get("status", "")
    if gw_status == "READY":
        break
    print(f"  Poll {i + 1}: {gw_status}")
    time.sleep(5)

GATEWAY_ARN = gw.get("gatewayArn") or f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}"
GATEWAY_URL = gw.get("gatewayUrl") or f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{REGION}.amazonaws.com"

print(f"\nGATEWAY_ID  = {GATEWAY_ID}")
print(f"GATEWAY_ARN = {GATEWAY_ARN}")
print(f"GATEWAY_URL = {GATEWAY_URL}")

In [ ]:
# v1 Agent Runtime을 가리키는 Gateway Target 생성(HTTP Target)
TARGET_NAME = "HRAgentV1"
tgt_resp = ctrl.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    description="HR Assistant v1 runtime target",
    targetConfiguration={
        "http": {
            "agentcoreRuntime": {
                "arn": AGENT_ARN,
                "qualifier": "DEFAULT",
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    clientToken=str(uuid.uuid4()),
)
TARGET_ID = tgt_resp["targetId"]
print(f"Target created: {TARGET_ID}. Polling for READY...")

for i in range(30):
    tgt = ctrl.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    tgt_status = tgt.get("status", "")
    if tgt_status == "READY":
        break
    print(f"  Poll {i + 1}: {tgt_status}")
    time.sleep(5)

print(f"TARGET_ID   = {TARGET_ID}")
print(f"TARGET_NAME = {TARGET_NAME}")

### 7b: Gateway 추적 설정

A/B 테스트에서는 각 세션이 어떤 그룹에 할당되었는지 알아야 합니다. 이 정보는 CloudWatch의 Gateway span에서 가져옵니다. X-Ray가 추적 세그먼트를 CloudWatch Logs로 전송하도록 설정한 다음, Gateway에서 X-Ray로 이어지는 전송 파이프라인을 생성합니다.

In [ ]:
# X-Ray가 추적 세그먼트를 CloudWatch Logs로 전송하도록 설정
dest = xray.get_trace_segment_destination()
if dest.get("Destination") != "CloudWatchLogs":
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("X-Ray trace destination set to CloudWatchLogs")
else:
    print("X-Ray trace destination already set to CloudWatchLogs")

# Gateway 추적용 전송 소스 생성
DELIVERY_SOURCE_NAME = f"hr-gw-traces-{SUFFIX}"
DELIVERY_ID = None
try:
    logs.put_delivery_source(
        name=DELIVERY_SOURCE_NAME,
        resourceArn=GATEWAY_ARN,
        logType="TRACES",
    )
    print(f"Delivery source created: {DELIVERY_SOURCE_NAME}")
except Exception as e:
    print(f"Delivery source: {e}")

# X-Ray 전송 대상 조회 또는 생성
destinations = logs.describe_delivery_destinations().get("deliveryDestinations", [])
xray_dest = next((d for d in destinations if d.get("deliveryDestinationType") == "XRAY"), None)
if not xray_dest:
    logs.put_delivery_destination(
        name="xray-destination",
        deliveryDestinationType="XRAY",
        deliveryDestinationConfiguration={
            "destinationResourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:group/Default"
        },
    )
    destinations = logs.describe_delivery_destinations().get("deliveryDestinations", [])
    xray_dest = next((d for d in destinations if d.get("deliveryDestinationType") == "XRAY"), None)

XRAY_DEST_ARN = xray_dest["arn"]
print(f"X-Ray delivery destination: {XRAY_DEST_ARN}")

# 전송 생성(소스 → 대상)
try:
    delivery = logs.create_delivery(
        deliverySourceName=DELIVERY_SOURCE_NAME,
        deliveryDestinationArn=XRAY_DEST_ARN,
    )
    DELIVERY_ID = delivery["delivery"]["id"]
    print(f"Delivery created: {DELIVERY_ID}")
except Exception as e:
    print(f"Delivery: {e}")
    # 이 소스의 기존 전송 조회
    for d in logs.describe_deliveries().get("deliveries", []):
        if d.get("deliverySourceName") == DELIVERY_SOURCE_NAME:
            DELIVERY_ID = d.get("id")
            print(f"Found existing delivery: {DELIVERY_ID}")
            break

print("Gateway tracing configured.")

### 7c: Online Evaluation Config 생성

**Online Evaluation Config**는 세션마다 명시적으로 API를 호출하지 않아도 Gateway를 통과하는 모든 세션의 점수를 자동으로 계산합니다. 에이전트의 CloudWatch 로그 그룹을 모니터링하고, `sessionTimeoutMinutes` 동안 활동이 없어서 세션이 종료된 시점을 감지한 후 설정된 평가기를 실행합니다.

In [ ]:
# Online Evaluation Config: 모든 세션 자동 평가
# Bundle 이름에는 영숫자와 underscore만 사용 가능(hyphen 불가)
online_eval_resp = ctrl.create_online_evaluation_config(
    onlineEvaluationConfigName=f"HROnlineEval{SUFFIX}",
    description="HR Assistant online evaluation for A/B testing",
    dataSourceConfig={
        "cloudWatchLogs": {
            "logGroupNames": [LOG_GROUP],
            "serviceNames": [SERVICE_NAME],
        }
    },
    evaluators=[
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.Helpfulness"},
    ],
    rule={
        "samplingConfig": {"samplingPercentage": 100.0},
        "sessionConfig": {"sessionTimeoutMinutes": 2},
    },
    evaluationExecutionRoleArn=ROLE_ARN,
    enableOnCreate=True,
    clientToken=str(uuid.uuid4()),
)
ONLINE_EVAL_ID = online_eval_resp["onlineEvaluationConfigId"]
ONLINE_EVAL_ARN = online_eval_resp["onlineEvaluationConfigArn"]
print(f"Online eval config: {ONLINE_EVAL_ID}")
print(f"ARN: {ONLINE_EVAL_ARN}")

### 7d: A/B 테스트 생성(Configuration Bundle 라우팅)

A/B 테스트는 Gateway에 라우팅 규칙을 설치합니다. 들어오는 각 요청은 `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` 헤더를 기준으로 그룹에 할당됩니다. 이 값을 제공하지 않으면 Runtime에서 자동으로 생성합니다. Gateway는 이 세션 ID와 설정된 트래픽 가중치를 사용하여 요청을 그룹에 할당합니다.
- **C (Control, 50%)**: Control Bundle을 통한 기존 시스템 프롬프트
- **T1 (Treatment, 50%)**: Treatment Bundle을 통한 추천 시스템 프롬프트

Gateway는 적절한 Configuration Bundle을 W3C baggage 헤더로 요청에 자동 주입합니다. 에이전트의 호출 핸들러는 `BedrockAgentCoreContext.get_config_bundle()`을 통해 이를 읽고 일치하는 프롬프트와 도구 설명을 적용합니다. 할당은 세션 내에서 고정됩니다.

In [ ]:
# A/B 테스트 이름에는 영숫자와 underscore만 사용 가능(hyphen 불가)
abtest_resp = dp.create_ab_test(
    name=f"HRBundleAB{SUFFIX}",
    description="HR Assistant: compare original vs recommended system prompt",
    gatewayArn=GATEWAY_ARN,
    roleArn=ROLE_ARN,
    enableOnCreate=True,
    evaluationConfig={"onlineEvaluationConfigArn": ONLINE_EVAL_ARN},
    variants=[
        {
            "name": "C",
            "weight": 50,
            "variantConfiguration": {
                "configurationBundle": {
                    "bundleArn": CONTROL_BUNDLE_ARN,
                    "bundleVersion": CONTROL_BUNDLE_VERSION,
                }
            },
        },
        {
            "name": "T1",
            "weight": 50,
            "variantConfiguration": {
                "configurationBundle": {
                    "bundleArn": TREATMENT_BUNDLE_ARN,
                    "bundleVersion": TREATMENT_BUNDLE_VERSION,
                }
            },
        },
    ],
    clientToken=str(uuid.uuid4()),
)
ABTEST_BUNDLE_ID = abtest_resp["abTestId"]
print(f"A/B test created: {ABTEST_BUNDLE_ID}")
print("Polling for ACTIVE/RUNNING...")

for i in range(30):
    ab = dp.get_ab_test(abTestId=ABTEST_BUNDLE_ID)
    s, es = ab.get("status", ""), ab.get("executionStatus", "")
    print(f"  Poll {i + 1}: status={s}  executionStatus={es}")
    if s == "ACTIVE" and es == "RUNNING":
        break
    if "FAILED" in s:
        print(f"  Error: {ab.get('errorDetails')}")
        break
    time.sleep(5)

print("\nA/B test LIVE. Gateway is splitting traffic 50/50 between C and T1.")

### 7e: Gateway를 통해 트래픽 전송

이제 Runtime에 직접 전송하지 않고 Gateway URL을 통해 트래픽을 전송합니다. Gateway는 세션 ID를 기준으로 각 세션을 그룹에 할당하고, baggage를 통해 해당 번들을 주입한 후 요청을 Runtime으로 전달합니다.

요청은 IAM 자격 증명을 사용하여 SigV4로 서명됩니다.

In [ ]:
import requests as http_requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

session = boto3.Session()
credentials = session.get_credentials().get_frozen_credentials()

# Gateway URL 형식: https://{gateway-id}.gateway.bedrock-agentcore.{region}.amazonaws.com/{target-name}/invocations
GW_INVOKE_URL = f"{GATEWAY_URL}/{TARGET_NAME}/invocations"

GW_PROMPTS = [
    "Employee ID: EMP-001. What is my current PTO balance?",
    "Employee ID: EMP-001. I need to request leave from 2026-08-04 to 2026-08-08 for a vacation.",
    "Employee ID: EMP-042. Can you explain our 401k matching policy?",
    "Employee ID: EMP-002. I only have a few days left. What exactly is the PTO rollover policy?",
    "Employee ID: EMP-001. Show me my January 2026 pay stub and explain the deductions.",
    "Employee ID: EMP-042. What are my health insurance options?",
    "Employee ID: EMP-001. What's the remote work policy at Acme?",
    "Employee ID: EMP-002. I need to take parental leave soon. How many weeks am I entitled to?",
    "Employee ID: EMP-042. Please submit a PTO request for 2026-09-01 to 2026-09-03 for personal reasons.",
    "Employee ID: EMP-001. How much life insurance does the company provide?",
    "Employee ID: EMP-001. Request time off from 2026-07-21 to 2026-07-25 for a family trip.",
    "Employee ID: EMP-042. What dental coverage do we have for major restorative work?",
    "Employee ID: EMP-002. I want to check my PTO balance before requesting leave.",
    "Employee ID: EMP-001. Can I work from home 4 days a week?",
    "Employee ID: EMP-042. What's the vision insurance allowance for contacts?",
    "Employee ID: EMP-001. Submit PTO for me: 2026-10-13 to 2026-10-14 for doctor appointments.",
    "Employee ID: EMP-002. Explain the 401k vesting schedule.",
    "Employee ID: EMP-042. What's the code of conduct policy around harassment?",
    "Employee ID: EMP-001. How much does the company contribute to health premiums for family coverage?",
    "Employee ID: EMP-042. Can you pull up my January 2026 pay stub?",
]

gw_session_ids = []
success, fail = 0, 0

for i, prompt in enumerate(GW_PROMPTS):
    sid = str(uuid.uuid4())
    gw_session_ids.append(sid)
    body = json.dumps({"prompt": prompt, "sessionId": sid})
    req = AWSRequest(
        method="POST",
        url=GW_INVOKE_URL,
        data=body,
        headers={
            "Content-Type": "application/json",
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": sid,
        },
    )
    SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(req)
    try:
        resp = http_requests.post(GW_INVOKE_URL, data=body, headers=dict(req.headers), timeout=120)
        if resp.status_code == 200:
            print(f"  [{i + 1:2d}/{len(GW_PROMPTS)}] OK  {sid[:8]}...  {resp.text[:80]}...")
            success += 1
        else:
            print(f"  [{i + 1:2d}/{len(GW_PROMPTS)}] ERR status={resp.status_code}: {resp.text[:100]}")
            fail += 1
    except Exception as e:
        print(f"  [{i + 1:2d}/{len(GW_PROMPTS)}] ERR {e}")
        fail += 1
    time.sleep(1)

print(f"\nTraffic summary: success={success}, fail={fail}, total={len(GW_PROMPTS)}")
print("Waiting for online evaluation pipeline...")

### 7f: A/B 테스트 결과 모니터링

다음 세 단계를 거치면 결과를 확인할 수 있습니다.
1. **세션 제한 시간**(2분): 평가기가 점수를 계산하기 전에 세션이 비활성화되기를 기다립니다.
2. **평가**(2~3분): 온라인 평가기가 각 세션의 점수를 계산합니다.
3. **집계**(약 5분 주기): Gateway span과 점수를 결합하여 그룹별 평균을 계산합니다.

마지막 요청 이후 10~15분 정도가 필요합니다. `analysisTimestamp` 값이 채워질 때까지 폴링합니다.

In [71]:
print("Polling for A/B test results (up to 20 minutes)...\n")
bundle_ab_results = None

for poll in range(25):
    ab = dp.get_ab_test(abTestId=ABTEST_BUNDLE_ID)
    results = ab.get("results", {})
    metrics = results.get("evaluatorMetrics", [])
    print(f"--- Poll {poll + 1}/25 -- {time.strftime('%H:%M:%S')} ---")
    print(f"  analysisTimestamp: {results.get('analysisTimestamp', 'none')}")

    for m in metrics:
        name = m.get("evaluatorArn", "").split("/")[-1]
        cs = m.get("controlStats", {})
        print(f"  Evaluator: {name}")
        print(f"    Control (C):   mean={cs.get('mean', '-')}  n={cs.get('sampleSize', '-')}")
        for vr in m.get("variantResults", []):
            # API에서 제공한 percentChange를 우선 사용하고, 없으면 직접 계산
            pct_change = vr.get("percentChange")
            if pct_change is None:
                cs_mean, vr_mean = cs.get("mean"), vr.get("mean")
                if cs_mean and vr_mean and float(cs_mean) != 0:
                    pct_change = (float(vr_mean) - float(cs_mean)) / float(cs_mean) * 100
            delta = f"  change={pct_change:+.1f}%" if pct_change is not None else ""
            print(
                f"    Treatment (T1): mean={vr.get('mean', '-')}  n={vr.get('sampleSize', '-')}  "
                f"p={vr.get('pValue', 'N/A')}  significant={vr.get('isSignificant', '-')}{delta}"
            )

    if results.get("analysisTimestamp") and metrics:
        bundle_ab_results = results
        print("\nResults are available!")
        break
    print()
    time.sleep(60)

if bundle_ab_results:
    print("\n" + "=" * 60)
    print("A/B TEST INTERPRETATION")
    print("=" * 60)
    for m in bundle_ab_results.get("evaluatorMetrics", []):
        name = m.get("evaluatorArn", "").split("/")[-1]
        cs_mean = m.get("controlStats", {}).get("mean")
        for vr in m.get("variantResults", []):
            sig = vr.get("isSignificant")
            t1_mean = vr.get("mean")
            p_value = vr.get("pValue", "N/A")
            pct_change = vr.get("percentChange")
            if pct_change is None and cs_mean and t1_mean and float(cs_mean) != 0:
                pct_change = (float(t1_mean) - float(cs_mean)) / float(cs_mean) * 100
            pct_str = f"{pct_change:+.1f}%" if pct_change is not None else "N/A"
            print(f"\n  {name}:  p-value={p_value}  change={pct_str}  significant={sig}")
            if sig and pct_change is not None and pct_change > 0:
                print("    RESULT: T1 wins (statistically significant improvement)")
                print("    ACTION: Promote treatment bundle as new default (Step 7g)")
            elif sig and pct_change is not None and pct_change < 0:
                print("    RESULT: T1 regressed (statistically significant decline)")
                print("    ACTION: Keep control; investigate recommendation quality")
            else:
                print("    RESULT: Inconclusive (p >= 0.05 or insufficient samples)")
                print("    ACTION: Continue sending traffic to accumulate sample size")

--- Poll 12/25 -- 17:12:32 ---
  analysisTimestamp: none

--- Poll 13/25 -- 17:13:32 ---
  analysisTimestamp: none

--- Poll 14/25 -- 17:14:32 ---
  analysisTimestamp: 2026-05-01 17:14:13.528000-07:00
  Evaluator: Builtin.GoalSuccessRate
    Control (C):   mean=0.9166666666666666  n=12
    Treatment (T1): mean=0.875  n=8  p=0.9555440006558459  significant=False  change=-4.5%
  Evaluator: Builtin.Helpfulness
    Control (C):   mean=0.8154545454545453  n=11
    Treatment (T1): mean=0.8112499999999999  n=8  p=0.9945452749178172  significant=False  change=-0.5%

Results are available!

A/B TEST INTERPRETATION

  Builtin.GoalSuccessRate:  p-value=0.9555440006558459  change=-4.5%  significant=False
    RESULT: Inconclusive (p >= 0.05 or insufficient samples)
    ACTION: Continue sending traffic to accumulate sample size

  Builtin.Helpfulness:  p-value=0.9945452749178172  change=-0.5%  significant=False
    RESULT: Inconclusive (p >= 0.05 or insufficient samples)
    ACTION: Continue sending

### 7g: 승격 — 우수한 구성으로 Control Bundle 업데이트

A/B 테스트에서 Treatment가 우수한 것으로 확인되면 Control Bundle을 추천 구성으로 업데이트하여 **승격**할 수 있습니다. 계보를 기록하도록 `parentVersionIds`를 전달하세요. 서비스는 올바른 상위 버전에서 업데이트하는지 확인하여 실수로 덮어쓰는 일을 방지합니다.


In [72]:
# ── Step 7g: Treatment 설정을 Control Bundle로 승격 ─────────────
# update_configuration_bundle은 기존 bundleId에 새 버전을 생성함

current = ctrl.get_configuration_bundle(bundleId=CONTROL_BUNDLE_ID)
CONTROL_BUNDLE_VERSION = current["versionId"]
print(f"Current control bundle version: {CONTROL_BUNDLE_VERSION}")

promote_resp = ctrl.update_configuration_bundle(
    bundleId=CONTROL_BUNDLE_ID,
    components={
        AGENT_ARN: {
            "configuration": {
                "system_prompt": RECOMMENDED_SYSTEM_PROMPT,
                "tool_descriptions": RECOMMENDED_TOOL_DESCRIPTIONS,
            }
        }
    },
    parentVersionIds=[CONTROL_BUNDLE_VERSION],  # 업데이트의 기준이 되는 현재 버전
    commitMessage="Promote treatment: AI-recommended prompt + tool descriptions (A/B validated)",
    clientToken=str(uuid.uuid4()),
)
CONTROL_BUNDLE_VERSION_V2 = promote_resp["versionId"]
print(f"Control bundle promoted to version: {CONTROL_BUNDLE_VERSION_V2}")
print(f"Previous version was             : {CONTROL_BUNDLE_VERSION}")
print()
print("The control bundle now carries the recommended config.")
print("All new sessions using this bundle will receive the improved prompt")
print("and tool descriptions — without any code redeployment.")

promoted_control_baggage = (
    f"aws.agentcore.configbundle_arn={CONTROL_BUNDLE_ARN},"
    f"aws.agentcore.configbundle_version={CONTROL_BUNDLE_VERSION_V2}"
)
print(f"\nUpdated baggage for promoted control:\n  {promoted_control_baggage}")

Current control bundle version: <VERSION_ID>
Control bundle promoted to version: <VERSION_ID>
Previous version was             : <VERSION_ID>

The control bundle now carries the recommended config.
All new sessions using this bundle will receive the improved prompt
and tool descriptions — without any code redeployment.

Updated baggage for promoted control:
  aws.agentcore.configbundle_arn=arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT_ID>:configuration-bundle/HRControl<HEX>-<ID>,aws.agentcore.configbundle_version=<VERSION_ID>


In [73]:
# Treatment Bundle 확인: 세션을 전송하여 에이전트가
# 기준 번들과 다르게 응답하는지 확인
treatment_baggage = (
    f"aws.agentcore.configbundle_arn={TREATMENT_BUNDLE_ARN},"
    f"aws.agentcore.configbundle_version={TREATMENT_BUNDLE_VERSION}"
)
control_baggage = (
    f"aws.agentcore.configbundle_arn={CONTROL_BUNDLE_ARN},aws.agentcore.configbundle_version={CONTROL_BUNDLE_VERSION}"
)

test_prompt = "Employee ID: EMP-001. I want to request 3 days off next week for a medical appointment."

ctrl_sid = str(uuid.uuid4())
ctrl_resp = dp.invoke_agent_runtime(
    agentRuntimeArn=AGENT_ARN,
    runtimeSessionId=ctrl_sid,
    payload=json.dumps({"prompt": test_prompt}).encode(),
    baggage=control_baggage,
)
ctrl_text = ctrl_resp["response"].read().decode("utf-8")

treat_sid = str(uuid.uuid4())
treat_resp = dp.invoke_agent_runtime(
    agentRuntimeArn=AGENT_ARN,
    runtimeSessionId=treat_sid,
    payload=json.dumps({"prompt": test_prompt}).encode(),
    baggage=treatment_baggage,
)
treat_text = treat_resp["response"].read().decode("utf-8")

print(f"Prompt: {test_prompt}\n")
print("[Control   ] " + ctrl_text[:300])
print()
print("[Treatment ] " + treat_text[:300])

Prompt: Employee ID: EMP-001. I want to request 3 days off next week for a medical appointment.

[Control   ] data: "<thinking"

data: ">"

data: " I"

data: " need"

data: " to"

data: " submit"

data: " a"

data: " PTO"

data: " request"

data: " for"

data: ""

data: " EMP"

data: "-"

data: "0"

data: "0"

data: "1"

data: "."

data: " I"

data: " will"

data: " need"

data: " the"

data: " employee"

d

[Treatment ] data: "<thinking"

data: ">"

data: "To"

data: " request"

data: ""

data: " 3"

data: " days"

data: " off"

data: ","

data: " I"

data: " first"

data: " need"

data: " to"

data: " check"

data: " the"

data: " employee"

data: "'"

data: "s"

data: " current"

data: " PTO"

data: " balance"

d


## Step 8: A/B 테스트 — Target-Based 라우팅(단계적 롤아웃)

테스트할 변경 사항에 코드 변경, 프레임워크 업그레이드 또는 완전히 다른 에이전트 구현이 포함되면 Target-Based 라우팅을 사용합니다. Target-Based 라우팅은 각각 다른 Gateway Target으로 등록된 두 개의 별도 Runtime에 트래픽을 전송합니다. Gateway는 A/B 테스트의 트래픽 가중치를 기준으로 각 세션을 두 Target 중 하나로 라우팅합니다.

다른 시스템 프롬프트, 모델 ID 또는 도구 설명처럼 구성만 변경하여 두 그룹을 같은 Runtime에서 실행할 수 있다면 Configuration Bundle 라우팅(Step 7)을 사용하세요.

**Configuration Bundle 라우팅과 Target-Based 라우팅 비교:**

| | Configuration Bundle 라우팅 | Target-Based 라우팅 |
|---|---|---|
| **변경 대상** | 시스템 프롬프트 또는 구성(코드 변경 없음) | 에이전트 코드, 도구 또는 모델 |
| **배포** | 재배포 불필요 | 새 Runtime 배포 필요 |
| **필요한 Runtime** | 공유 Runtime 하나 | 별도 Runtime 두 개 |
| **필요한 평가 구성** | 공유 Online Evaluation Config 하나 | 그룹별 하나(서로 다른 로그 그룹) |
| **사용 사례** | 프롬프트 최적화, 구성 튜닝 | 코드 롤아웃, 버전 업그레이드 |
| **위험** | 매우 낮음, 번들로 즉시 롤백 | 더 높음, 바이너리 변경 |

세션 할당은 **고정(sticky)**됩니다. 세션 ID가 그룹에 할당되면 이후 같은 세션 ID의 모든 요청이 동일한 Target Runtime으로 라우팅됩니다. 따라서 세션 내에서 일관된 동작을 보장하면서 새 세션은 트래픽 가중치에 따라 분산할 수 있습니다.

**사용 사례: HR Assistant v2의 단계적 롤아웃**

v2에는 새 `escalate_to_hr_manager` 도구와 코드에 포함된 더 상세한 시스템 프롬프트가 추가됩니다. 모든 사용자에게 즉시 100% 롤아웃하는 대신 다음과 같이 진행합니다.
1. v1과 함께 v2 배포(두 Runtime 모두 실행)
2. 트래픽의 10%를 v2로 라우팅(카나리)
3. 각 Runtime을 자체 Online Evaluation Config로 평가
4. 지표가 개선되면 50%, 이어서 100%로 승격

### 8a: HR Assistant v2 배포

In [74]:
# HR Assistant v2 배포: 새 도구 + 개선된 시스템 프롬프트(코드 변경 시뮬레이션)
result = subprocess.run(
    [
        sys.executable,
        "deploy_agent.py",
        "--name",
        V2_NAME,
        "--region",
        REGION,
        "--version",
        "v2",
    ],
    capture_output=False,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"deploy_agent.py (v2) failed with exit code {result.returncode}")

Deploying HRAssistV2<HEX> (version=v2) to us-east-1 (account=<ACCOUNT_ID>)
Created IAM role: arn:aws:iam::<ACCOUNT_ID>:role/HRAssistV2<HEX>Role
IAM policy attached. Waiting 10s for propagation...
Installing dependencies for ARM64 into /tmp/HRAssistV2<HEX>_build/pkg...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openapi-spec-validator 0.8.4 requires jsonschema<4.25.0,>=4.24.0, but you have jsonschema 4.26.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


Agent code (v2) written to /tmp/HRAssistV2<HEX>_build/pkg/main.py
Package built: /tmp/HRAssistV2<HEX>_build/deployment_package.zip (34.2 MB)
Created S3 bucket: bedrock-agentcore-code-<ACCOUNT_ID>-us-east-1
Uploaded to s3://bedrock-agentcore-code-<ACCOUNT_ID>-us-east-1/HRAssistV2<HEX>/deployment_package.zip
Runtime created: HRAssistV2<HEX>-<ID>. Polling for READY/ACTIVE...
  Poll 1: CREATING
  Poll 2: CREATING
  Poll 3: READY

State saved to agent_state_HRAssistV2<HEX>.json
{
  "runtime_name": "HRAssistV2<HEX>",
  "runtime_id": "HRAssistV2<HEX>-<ID>",
  "runtime_arn": "arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT_ID>:runtime/HRAssistV2<HEX>-<ID>",
  "log_group": "/aws/bedrock-agentcore/runtimes/HRAssistV2<HEX>-<ID>-DEFAULT",
  "service_name": "HRAssistV2<HEX>.DEFAULT",
  "role_arn": "arn:aws:iam::<ACCOUNT_ID>:role/HRAssistV2<HEX>Role",
  "role_name": "HRAssistV2<HEX>Role",
  "s3_bucket": "bedrock-agentcore-code-<ACCOUNT_ID>-us-east-1",
  "s3_key": "HRAssistV2<HEX>/deployment_package.zip

In [75]:
# v2 상태 불러오기
v2_state = json.loads(Path(f"agent_state_{V2_NAME}.json").read_text())

AGENT_ARN_V2 = v2_state["runtime_arn"]
AGENT_ID_V2 = v2_state["runtime_id"]
LOG_GROUP_V2 = v2_state["log_group"]
SERVICE_NAME_V2 = v2_state["service_name"]
ROLE_ARN_V2 = v2_state["role_arn"]

print(f"v2 Agent ARN : {AGENT_ARN_V2}")
print(f"v2 Log Group : {LOG_GROUP_V2}")
print("v2 has extra tool: escalate_to_hr_manager")
print("v2 has improved system prompt baked into the code")

v2 Agent ARN : arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT_ID>:runtime/HRAssistV2<HEX>-<ID>
v2 Log Group : /aws/bedrock-agentcore/runtimes/HRAssistV2<HEX>-<ID>-DEFAULT
v2 has extra tool: escalate_to_hr_manager
v2 has improved system prompt baked into the code


### 8b: v2 Gateway Target 추가

기존 Gateway에 두 번째 Target을 추가합니다. 이제 Gateway에는 두 Target이 있습니다.
- `HRAgentV1`: v1 Runtime을 가리킴(안정적인 기준)
- `HRAgentV2`: v2 Runtime을 가리킴(테스트 중인 새 코드)

Target-Based A/B 테스트는 이름으로 이 Target들을 참조합니다.

In [76]:
# v2 Runtime을 가리키는 두 번째 Gateway Target 생성
TARGET_NAME_V2 = "HRAgentV2"
tgt_v2_resp = ctrl.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME_V2,
    description="HR Assistant v2 runtime target (new escalation tool + improved prompt)",
    targetConfiguration={
        "http": {
            "agentcoreRuntime": {
                "arn": AGENT_ARN_V2,
                "qualifier": "DEFAULT",
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    clientToken=str(uuid.uuid4()),
)
TARGET_ID_V2 = tgt_v2_resp["targetId"]
print(f"v2 target created: {TARGET_ID_V2}. Polling for READY...")

for i in range(30):
    tgt_v2 = ctrl.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID_V2)
    if tgt_v2.get("status") == "READY":
        break
    print(f"  Poll {i + 1}: {tgt_v2.get('status')}")
    time.sleep(5)

print("\nGateway now has two targets:")
print(f"  {TARGET_NAME}    -> v1 runtime (stable)")
print(f"  {TARGET_NAME_V2} -> v2 runtime (canary with new tool)")

v2 target created: BPHW22F3O4. Polling for READY...
  Poll 1: CREATING

Gateway now has two targets:
  HRAgentV1    -> v1 runtime (stable)
  HRAgentV2 -> v2 runtime (canary with new tool)


### 8c: v2용 Online Evaluation Config 생성

Target-Based 라우팅에는 v2 로그 그룹을 모니터링하는 두 번째 Online Evaluation Config가 필요합니다. 각 그룹의 세션이 자체 구성으로 평가되도록 `perVariantOnlineEvaluationConfig`를 사용합니다.

In [77]:
# v2 세션용 Online Evaluation Config
online_eval_v2_resp = ctrl.create_online_evaluation_config(
    onlineEvaluationConfigName=f"HROnlineEvalV2{SUFFIX}",
    description="HR Assistant v2 online evaluation (target-based routing)",
    dataSourceConfig={
        "cloudWatchLogs": {
            "logGroupNames": [LOG_GROUP_V2],
            "serviceNames": [SERVICE_NAME_V2],
        }
    },
    evaluators=[
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.Helpfulness"},
    ],
    rule={
        "samplingConfig": {"samplingPercentage": 100.0},
        "sessionConfig": {"sessionTimeoutMinutes": 2},
    },
    evaluationExecutionRoleArn=ROLE_ARN_V2,
    enableOnCreate=True,
    clientToken=str(uuid.uuid4()),
)
ONLINE_EVAL_V2_ID = online_eval_v2_resp["onlineEvaluationConfigId"]
ONLINE_EVAL_V2_ARN = online_eval_v2_resp["onlineEvaluationConfigArn"]
print(f"v2 online eval config: {ONLINE_EVAL_V2_ID}")

v2 online eval config: HROnlineEvalV2<HEX>-<ID>


### 8d: Target-Based A/B 테스트 생성(90/10 카나리 분할)

`variantConfiguration`에서 `configurationBundle` 대신 `target: {name: ...}`을 지정합니다. Gateway는 각 세션을 이름으로 지정한 Target에 라우팅하며, 해당 Target은 특정 Runtime ARN에 매핑됩니다.

일반적인 카나리 롤아웃 방식인 **90/10 분할**을 사용합니다.
- **C (90%)**: v1 Runtime — 안정적인 프로덕션
- **T1 (10%)**: v2 Runtime — 새 기능을 포함한 카나리

각 그룹에 자체 로그 그룹이 있으므로(로그 그룹 이름에 엔드포인트 이름 포함) `perVariantOnlineEvaluationConfig`를 사용합니다.

`gatewayFilter.targetPaths`는 A/B 테스트 라우팅 규칙의 범위를 Control Target 경로와 일치하는 요청으로 제한하여 이 테스트용 트래픽만 가로채도록 합니다.

`enableOnCreate=True`는 생성 직후 테스트를 시작하므로 별도의 `update_ab_test` 호출이 필요하지 않습니다.

In [78]:
# Gateway마다 한 번에 하나의 A/B 테스트만 실행 가능
# Target 라우팅 테스트를 생성하기 전에 Configuration Bundle A/B 테스트 중지
print("Stopping config-bundle A/B test to free the gateway for target routing...")
try:
    ab = dp.get_ab_test(abTestId=ABTEST_BUNDLE_ID)
    if ab.get("executionStatus") in ("RUNNING", "PAUSED"):
        dp.update_ab_test(abTestId=ABTEST_BUNDLE_ID, executionStatus="STOPPED")
        for i in range(20):
            ab = dp.get_ab_test(abTestId=ABTEST_BUNDLE_ID)
            if ab.get("executionStatus") == "STOPPED":
                print("  Bundle A/B test stopped (executionStatus=STOPPED)")
                break
            time.sleep(5)
    else:
        print(f"  Bundle A/B test already in state: {ab.get('executionStatus')}")
except Exception as e:
    print(f"  Stop skipped: {e}")

# Target-Based A/B 테스트: Gateway Target 이름으로 라우팅(서로 다른 Runtime)
# gatewayFilter.targetPaths는 라우팅 규칙의 범위를 Control Target 경로로 제한
# enableOnCreate=True는 별도의 update_ab_test 없이 테스트를 즉시 시작
abtest_target_resp = dp.create_ab_test(
    name=f"HRTargetAB{SUFFIX}",
    description="HR Assistant: phased rollout v1 (90%) vs v2 (10%) — target-based routing",
    gatewayArn=GATEWAY_ARN,
    roleArn=ROLE_ARN,
    enableOnCreate=True,
    evaluationConfig={
        "perVariantOnlineEvaluationConfig": [
            {"name": "C", "onlineEvaluationConfigArn": ONLINE_EVAL_ARN},
            {"name": "T1", "onlineEvaluationConfigArn": ONLINE_EVAL_V2_ARN},
        ]
    },
    gatewayFilter={"targetPaths": [f"/{TARGET_NAME}/*"]},
    variants=[
        {
            "name": "C",
            "weight": 90,
            "variantConfiguration": {
                "target": {"name": TARGET_NAME}  # v1 Runtime(안정적)
            },
        },
        {
            "name": "T1",
            "weight": 10,
            "variantConfiguration": {
                "target": {"name": TARGET_NAME_V2}  # v2 Runtime(카나리)
            },
        },
    ],
    clientToken=str(uuid.uuid4()),
)
ABTEST_TARGET_ID = abtest_target_resp["abTestId"]
print(f"Target-based A/B test created: {ABTEST_TARGET_ID}")
print("Polling for ACTIVE/RUNNING...")

for i in range(30):
    ab = dp.get_ab_test(abTestId=ABTEST_TARGET_ID)
    s, es = ab.get("status", ""), ab.get("executionStatus", "")
    print(f"  Poll {i + 1}: status={s}  executionStatus={es}")
    if s == "ACTIVE" and es == "RUNNING":
        break
    if "FAILED" in s:
        print(f"  Error: {ab.get('errorDetails')}")
        raise RuntimeError("Failed to create/start target A/B test")
    time.sleep(5)

print("\nTarget-based A/B test LIVE.")
print(f"  90% of traffic -> {TARGET_NAME}  (v1, stable)")
print(f"  10% of traffic -> {TARGET_NAME_V2} (v2, canary)")

Stopping config-bundle A/B test to free the gateway for target routing...
  Bundle A/B test stopped (executionStatus=STOPPED)
Target-based A/B test created: hrtargetab<HEX>-<ID>
Polling for ACTIVE/RUNNING...
  Poll 1: status=CREATING  executionStatus=NOT_STARTED
  Poll 2: status=ACTIVE  executionStatus=RUNNING

Target-based A/B test LIVE.
  90% of traffic -> HRAgentV1  (v1, stable)
  10% of traffic -> HRAgentV2 (v2, canary)


### 8e: Target-Based A/B 테스트로 트래픽 전송

동일한 Gateway URL을 통해 트래픽을 전송합니다. 이제 Target-Based A/B 테스트가 라우팅 규칙을 재정의하여 90%는 v1, 10%는 v2로 전달합니다.

In [79]:
# Target-Based A/B 테스트가 Target 이름을 기준으로 라우팅하므로 v2 Target URL 사용
# 두 Target URL 중 어느 쪽으로 보낸 요청이든 A/B 테스트 라우팅 규칙에 따라 분할됨
GW_INVOKE_URL_V2 = f"{GATEWAY_URL}/{TARGET_NAME_V2}/invocations"

# Target-Based 라우팅 트래픽을 생성하기 위한 다양한 HR 프롬프트
TARGET_PROMPTS = [
    "Employee ID: EMP-001. Check my PTO balance and submit a request for 2026-11-24 to 2026-11-28.",
    "Employee ID: EMP-042. I have a payroll dispute. Can you escalate this to an HR manager?",
    "Employee ID: EMP-002. What benefits can I enroll in during open enrollment?",
    "Employee ID: EMP-001. What's the maximum PTO carryover allowed?",
    "Employee ID: EMP-042. My manager is creating a hostile work environment. I need help.",
    "Employee ID: EMP-001. How many weeks of parental leave will I get as a primary caregiver?",
    "Employee ID: EMP-002. Pull up my pay stub for January 2026.",
    "Employee ID: EMP-001. Can I take PTO before I've fully accrued the days?",
    "Employee ID: EMP-042. I need a dental claim reviewed — can you escalate?",
    "Employee ID: EMP-001. What vision insurance benefits do we have?",
]

target_session_ids = []
success, fail = 0, 0

for i, prompt in enumerate(TARGET_PROMPTS):
    sid = str(uuid.uuid4())
    target_session_ids.append(sid)
    # 다양한 트래픽을 생성하도록 두 Target URL에 번갈아 전송
    invoke_url = GW_INVOKE_URL if i % 2 == 0 else GW_INVOKE_URL_V2
    body = json.dumps({"prompt": prompt, "sessionId": sid})
    req = AWSRequest(
        method="POST",
        url=invoke_url,
        data=body,
        headers={
            "Content-Type": "application/json",
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": sid,
        },
    )
    SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(req)
    try:
        resp = http_requests.post(invoke_url, data=body, headers=dict(req.headers), timeout=120)
        if resp.status_code == 200:
            print(f"  [{i + 1:2d}/{len(TARGET_PROMPTS)}] OK  {sid[:8]}...  {resp.text[:80]}...")
            success += 1
        else:
            print(f"  [{i + 1:2d}/{len(TARGET_PROMPTS)}] ERR status={resp.status_code}: {resp.text[:100]}")
            fail += 1
    except Exception as e:
        print(f"  [{i + 1:2d}/{len(TARGET_PROMPTS)}] ERR {e}")
        fail += 1
    time.sleep(1)

print(f"\nTraffic summary: success={success}, fail={fail}, total={len(TARGET_PROMPTS)}")
print("Waiting for online evaluation pipeline...")

  [ 1/10] OK  c87c8e7e...  data: "<thinking"

data: ">"

data: "The"

data: " user"

data: " has"

data: " ...
  [ 2/10] OK  3e5fcb4f...  data: "<thinking"

data: ">"

data: "The"

data: " employee"

data: " has"

data...
  [ 3/10] OK  ec93378a...  data: "<thinking"

data: ">"

data: "I"

data: " need"

data: " to"

data: " ret...
  [ 4/10] OK  27a598a4...  data: "<thinking"

data: ">"

data: "The"

data: " employee"

data: " has"

data...
  [ 5/10] OK  6cd1da26...  data: "<thinking"

data: ">"

data: "The"

data: " User"

data: " is"

data: " e...
  [ 6/10] OK  88ab7e43...  data: "<thinking"

data: ">"

data: "The"

data: " employee"

data: " has"

data...
  [ 7/10] OK  d4224c40...  data: "<thinking"

data: ">"

data: "The"

data: " employee"

data: " has"

data...
  [ 8/10] OK  4306d4c3...  data: "<thinking"

data: ">"

data: "The"

data: " employee"

data: " is"

data:...
  [ 9/10] OK  db3d2bcf...  data: "<thinking"

data: ">"

data: "The"

data: " request"

data: " is"

data: ...
 

### 8f: Target-Based A/B 테스트 결과 모니터링

In [80]:
print("Polling for target-based A/B test results (up to 20 minutes)...\n")
target_ab_results = None

for poll in range(25):
    ab = dp.get_ab_test(abTestId=ABTEST_TARGET_ID)
    results = ab.get("results", {})
    metrics = results.get("evaluatorMetrics", [])
    print(f"--- Poll {poll + 1}/25 -- {time.strftime('%H:%M:%S')} ---")
    print(f"  analysisTimestamp: {results.get('analysisTimestamp', 'none')}")

    for m in metrics:
        name = m.get("evaluatorArn", "").split("/")[-1]
        cs = m.get("controlStats", {})
        print(f"  Evaluator: {name}")
        print(f"    v1 Control  (C,  90%): mean={cs.get('mean', '-')}  n={cs.get('sampleSize', '-')}")
        for vr in m.get("variantResults", []):
            # API에서 제공한 percentChange를 우선 사용하고, 없으면 직접 계산
            pct_change = vr.get("percentChange")
            if pct_change is None:
                cs_mean, vr_mean = cs.get("mean"), vr.get("mean")
                if cs_mean and vr_mean and float(cs_mean) != 0:
                    pct_change = (float(vr_mean) - float(cs_mean)) / float(cs_mean) * 100
            delta = f"  change={pct_change:+.1f}%" if pct_change is not None else ""
            print(
                f"    v2 Treatment (T1, 10%): mean={vr.get('mean', '-')}  n={vr.get('sampleSize', '-')}  "
                f"p={vr.get('pValue', 'N/A')}  significant={vr.get('isSignificant', '-')}{delta}"
            )

    if results.get("analysisTimestamp") and metrics:
        target_ab_results = results
        print("\nResults are available!")
        break
    print()
    time.sleep(60)

if target_ab_results:
    print("\n" + "=" * 60)
    print("TARGET-BASED ROUTING INTERPRETATION")
    print("=" * 60)
    for m in target_ab_results.get("evaluatorMetrics", []):
        name = m.get("evaluatorArn", "").split("/")[-1]
        cs_mean = m.get("controlStats", {}).get("mean")
        for vr in m.get("variantResults", []):
            sig = vr.get("isSignificant")
            t1_mean = vr.get("mean")
            p_value = vr.get("pValue", "N/A")
            pct_change = vr.get("percentChange")
            if pct_change is None and cs_mean and t1_mean and float(cs_mean) != 0:
                pct_change = (float(t1_mean) - float(cs_mean)) / float(cs_mean) * 100
            pct_str = f"{pct_change:+.1f}%" if pct_change is not None else "N/A"
            print(f"\n  {name}:  p-value={p_value}  change={pct_str}  significant={sig}")
            if sig and pct_change is not None and pct_change > 0:
                print("    RESULT: v2 wins (statistically significant improvement)")
                print("    ACTION: Ramp to 50%, then 100% cutover to v2")
            elif sig and pct_change is not None and pct_change < 0:
                print("    RESULT: v2 regressed (statistically significant decline)")
                print("    ACTION: Halt rollout; keep v1, investigate v2")
            else:
                print("    RESULT: Inconclusive (p >= 0.05 or insufficient samples)")
                print("    ACTION: Continue sending traffic to accumulate sample size")
    print("\nPhased rollout workflow:")
    print("  10% canary  -> validate no regressions")
    print("  50% ramp    -> gather statistical significance")
    print("  100% promote -> complete cutover to v2")
    print("  To update weights: dp.update_ab_test(abTestId=ABTEST_TARGET_ID, variants=[...new weights...])")

Polling for target-based A/B test results (up to 20 minutes)...

--- Poll 1/25 -- 17:18:14 ---
  analysisTimestamp: 2026-05-01 17:17:31.565000-07:00
  Evaluator: Builtin.GoalSuccessRate
    v1 Control  (C,  90%): mean=0.9166666666666666  n=12
    v2 Treatment (T1, 10%): mean=0.875  n=8  p=0.9555440006558459  significant=False  change=-4.5%
  Evaluator: Builtin.Helpfulness
    v1 Control  (C,  90%): mean=0.8154545454545453  n=11
    v2 Treatment (T1, 10%): mean=0.8112499999999999  n=8  p=0.9945452749178172  significant=False  change=-0.5%

Results are available!

TARGET-BASED ROUTING INTERPRETATION

  Builtin.GoalSuccessRate:  p-value=0.9555440006558459  change=-4.5%  significant=False
    RESULT: Inconclusive (p >= 0.05 or insufficient samples)
    ACTION: Continue sending traffic to accumulate sample size

  Builtin.Helpfulness:  p-value=0.9945452749178172  change=-0.5%  significant=False
    RESULT: Inconclusive (p >= 0.05 or insufficient samples)
    ACTION: Continue sending traffic

## 요약

전체 AgentCore Optimization 워크플로를 완료했습니다.

| 단계 | 수행한 작업 | 주요 API |
|------|-------------|---------|
| 2 | HR Assistant를 AgentCore Runtime에 배포 | `create_agent_runtime` |
| 3 | 기준 Configuration Bundle 생성 및 트래픽 전송 | `create_configuration_bundle`, `invoke_agent_runtime` |
| 4 | 기준 성능 측정 | `start_batch_evaluation` / `get_batch_evaluation` |
| 5a | 추적 데이터에서 개선된 시스템 프롬프트 생성 | `start_recommendation` (SYSTEM_PROMPT) |
| 5b | 추적 데이터에서 개선된 도구 설명 생성 | `start_recommendation` (TOOL_DESCRIPTION) |
| 6 | Control 및 Treatment 구성(시스템 프롬프트와 도구 설명) 패키징 | `create_configuration_bundle` |
| 7 | Configuration Bundle 라우팅으로 프롬프트 및 도구 설명 변경 사항 A/B 테스트 | `create_ab_test` (configurationBundle 그룹) |
| 8 | Target-Based 라우팅으로 v2 카나리 롤아웃 | `create_ab_test` (Target 그룹, 90/10 분할) |

### 의사 결정 기준

**Configuration Bundle A/B 테스트에서 T1이 우수한가요?** → 기준 번들을 Treatment 구성(추천 시스템 프롬프트 + 도구 설명)으로 업데이트합니다. 코드 배포 없이 새 기본 구성이 됩니다.

**Target-Based A/B 테스트에서 v2가 우수한가요?** → A/B 테스트 가중치를 50/50으로 업데이트한 다음 100/0으로 변경합니다. v2가 100%가 되면 v1 Runtime을 삭제합니다.

**두 테스트 중 하나에서 회귀가 나타났나요?** → A/B 테스트를 중지하고(`update_ab_test(executionStatus='STOPPED')`) 원인을 조사한 후 반복 개선합니다.

## Step 9: 정리

이 노트북에서 생성한 모든 AWS 리소스를 삭제합니다. 각 블록은 독립적으로 실행되므로 일부만 실행한 경우에도 가능한 리소스를 정리합니다.

In [81]:
# ── 1. A/B 테스트 중지 및 삭제 ──────────────────────────────────────────
for ab_id, label in [
    (ABTEST_BUNDLE_ID if "ABTEST_BUNDLE_ID" in dir() else None, "bundle"),
    (ABTEST_TARGET_ID if "ABTEST_TARGET_ID" in dir() else None, "target"),
]:
    if not ab_id:
        continue
    print(f"1. Deleting A/B test ({label}): {ab_id}")
    try:
        # 삭제 전에 테스트 중지(STOPPED는 트래픽 라우팅을 즉시 종료)
        ab = dp.get_ab_test(abTestId=ab_id)
        if ab.get("executionStatus") in ("RUNNING", "PAUSED"):
            dp.update_ab_test(abTestId=ab_id, executionStatus="STOPPED")
            time.sleep(3)
        dp.delete_ab_test(abTestId=ab_id)
        print(f"   Deleted: {ab_id}")
    except Exception as e:
        print(f"   Skipped: {e}")

# ── 2. Online Evaluation Config 삭제 ───────────────────────────────────
for oe_id, label in [
    (ONLINE_EVAL_ID if "ONLINE_EVAL_ID" in dir() else None, "v1"),
    (ONLINE_EVAL_V2_ID if "ONLINE_EVAL_V2_ID" in dir() else None, "v2"),
]:
    if not oe_id:
        continue
    print(f"2. Deleting online eval config ({label}): {oe_id}")
    try:
        # 삭제 전에 비활성화(enableOnCreate=False가 아닌 executionStatus='DISABLED' 사용)
        ctrl.update_online_evaluation_config(onlineEvaluationConfigId=oe_id, executionStatus="DISABLED")
        time.sleep(2)
        ctrl.delete_online_evaluation_config(onlineEvaluationConfigId=oe_id)
        print(f"   Deleted: {oe_id}")
    except Exception as e:
        print(f"   Skipped: {e}")

# ── 3. Configuration Bundle 삭제 ───────────────────────────────────────
for b_id, label in [
    (BASELINE_BUNDLE_ID if "BASELINE_BUNDLE_ID" in dir() else None, "baseline"),
    (CONTROL_BUNDLE_ID if "CONTROL_BUNDLE_ID" in dir() else None, "control"),
    (TREATMENT_BUNDLE_ID if "TREATMENT_BUNDLE_ID" in dir() else None, "treatment"),
]:
    if not b_id:
        continue
    print(f"3. Deleting bundle ({label}): {b_id}")
    try:
        ctrl.delete_configuration_bundle(bundleId=b_id)
        print(f"   Deleted: {b_id}")
    except Exception as e:
        print(f"   Skipped: {e}")

# ── 4. Gateway 추적 삭제(전송 + 소스) ─────────────────────────
if "DELIVERY_ID" in dir() and DELIVERY_ID:
    print("4a. Deleting delivery...")
    try:
        logs.delete_delivery(id=DELIVERY_ID)
        print(f"   Deleted delivery: {DELIVERY_ID}")
    except Exception as e:
        print(f"   Skipped delivery: {e}")

if "DELIVERY_SOURCE_NAME" in dir():
    print("4b. Deleting delivery source...")
    try:
        logs.delete_delivery_source(name=DELIVERY_SOURCE_NAME)
        print(f"   Deleted delivery source: {DELIVERY_SOURCE_NAME}")
    except Exception as e:
        print(f"   Skipped delivery source: {e}")

# ── 5. Gateway Target 및 Gateway 삭제 ───────────────────────────────────
if "GATEWAY_ID" in dir():
    for t_id, tname in [
        (TARGET_ID_V2 if "TARGET_ID_V2" in dir() else None, "v2"),
        (TARGET_ID if "TARGET_ID" in dir() else None, "v1"),
    ]:
        if not t_id:
            continue
        print(f"5. Deleting gateway target ({tname}): {t_id}")
        try:
            ctrl.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=t_id)
            time.sleep(3)
            print(f"   Deleted: {t_id}")
        except Exception as e:
            print(f"   Skipped: {e}")

1. Deleting A/B test (bundle): hrbundleab<HEX>-<ID>
   Deleted: hrbundleab<HEX>-<ID>
1. Deleting A/B test (target): hrtargetab<HEX>-<ID>
   Deleted: hrtargetab<HEX>-<ID>
2. Deleting online eval config (v1): HROnlineEval<HEX>-<ID>
   Deleted: HROnlineEval<HEX>-<ID>
2. Deleting online eval config (v2): HROnlineEvalV2<HEX>-<ID>
   Deleted: HROnlineEvalV2<HEX>-<ID>
3. Deleting bundle (baseline): HRBaseline<HEX>-<ID>
   Deleted: HRBaseline<HEX>-<ID>
3. Deleting bundle (control): HRControl<HEX>-<ID>
   Deleted: HRControl<HEX>-<ID>
3. Deleting bundle (treatment): HRTreatment<HEX>-<ID>
   Deleted: HRTreatment<HEX>-<ID>
4a. Deleting delivery...
   Deleted delivery: PPcfgizaO3ZNJiwh
4b. Deleting delivery source...
   Deleted delivery source: hr-gw-traces-<HEX>
5. Deleting gateway target (v2): BPHW22F3O4
   Deleted: BPHW22F3O4
5. Deleting gateway target (v1): T6GWMBAMRZ
   Deleted: T6GWMBAMRZ
